# Nectar Data Engineer Challenge
## End-to-End IoT Data Platform Implementation

This notebook demonstrates the complete implementation of the Nectar Data Engineer Challenge, covering:
1. **Ingestion & Data Quality Framework:** Load telemetry/event datasets and validate schemas, nulls, duplicates, outliers, and late-arriving logs.
2. **Data Modeling & ETL:** Store structured dimensions and facts into a SQLite database with gold-level aggregates.
3. **SQL Challenge Execution:** Execute CTE and window-function based queries to extract key operational insights.
4. **Multi-Asset Hierarchy Analysis:** Build directed graphs with NetworkX to model site/building asset topologies and simulate failure impacts.

In [ ]:
import os
import pandas as pd
import numpy as np
import sqlite3
import networkx as nx
import json

# Ensure we are in the correct working directory
print("Current directory:", os.getcwd())

### 1. Ingestion & Data Quality Framework

We ingest raw IoT sensor telemetry and operational event files, then pass them through the `DataQualityFramework` to flag discrepancies before database injection.

In [ ]:
from src.data_quality import DataQualityFramework

assets_csv = "data/assets_metadata.csv"
telemetry_csv = "data/telemetry_raw.csv"
events_csv = "data/events_raw.csv"

# Execute data quality validation
dq = DataQualityFramework(assets_csv, telemetry_csv, events_csv)
report = dq.run_checks()

# Print data quality summary
print(json.dumps(report["summary"], indent=4))

### 2. ETL & Analytical Database Modeling

We clean the raw data (deduplicate, filter out invalid asset references/outliers) and populate our Star Schema in the SQLite database.

In [ ]:
from src.models import init_db
from src.pipeline import run_pipeline

# Initialize database schema
init_db("nectar_iot.db")

# Execute the pipeline to clean, model, and load data
run_pipeline(assets_csv, telemetry_csv, events_csv, "nectar_iot.db")

### 3. SQL Challenge Live Execution

Let's execute the SQL queries developed to solve the 6 challenge questions.

In [ ]:
conn = sqlite3.connect("nectar_iot.db")

# Question 1: Top 10 assets with the highest energy consumption
q1 = """
SELECT 
    fe.asset_id,
    da.asset_name,
    ROUND(SUM(fe.power_consumption) * 0.25, 2) as total_energy_kwh
FROM fact_energy fe
JOIN dim_asset da ON fe.asset_id = da.asset_id
GROUP BY fe.asset_id, da.asset_name
ORDER BY total_energy_kwh DESC
LIMIT 10;
"""
df_q1 = pd.read_sql_query(q1, conn)
print("\n--- 1. Top 10 Energy Consuming Assets ---")
print(df_q1.to_string(index=False))

In [ ]:
# Question 2: Calculate average daily energy consumption for each site
q2 = """
WITH daily_site_energy AS (
    SELECT 
        site_id,
        DATE(timestamp) as date_val,
        SUM(power_consumption) * 0.25 as daily_energy
    FROM fact_energy
    WHERE site_id IS NOT NULL AND site_id != ''
    GROUP BY site_id, DATE(timestamp)
SELECT 
    site_id,
    ROUND(AVG(daily_energy), 2) as avg_daily_energy_kwh
FROM daily_site_energy
GROUP BY site_id;
"""
df_q2 = pd.read_sql_query(q2, conn)
print("\n--- 2. Average Daily Energy per Site ---")
print(df_q2.to_string(index=False))

In [ ]:
# Question 3: Identify assets that generated more than 10 faults in the last 30 days
q3 = """
SELECT 
    fe.asset_id,
    da.asset_name,
    COUNT(fe.event_id) as fault_count
FROM fact_event fe
JOIN dim_asset da ON fe.asset_id = da.asset_id
WHERE fe.event_type = 'Fault'
  AND datetime(fe.timestamp) >= datetime('now', '-30 days')
GROUP BY fe.asset_id, da.asset_name
HAVING fault_count > 10;
"""
df_q3 = pd.read_sql_query(q3, conn)
print("\n--- 3. Assets with >10 Faults in Last 30 Days ---")
print(df_q3.to_string(index=False))

In [ ]:
# Question 4: Find assets that have not reported telemetry for the last 24 hours
q4 = """
WITH max_db_time AS (
    SELECT MAX(timestamp) as max_time FROM fact_telemetry
SELECT 
    da.asset_id,
    da.asset_name,
    MAX(ft.timestamp) as last_reported_time
FROM dim_asset da
LEFT JOIN fact_telemetry ft ON da.asset_id = ft.asset_id
CROSS JOIN max_db_time mdt
GROUP BY da.asset_id, da.asset_name
HAVING last_reported_time IS NULL 
   OR datetime(last_reported_time) < datetime(mdt.max_time, '-24 hours');
"""
df_q4 = pd.read_sql_query(q4, conn)
print("\n--- 4. Silent Assets (No telemetry in last 24 hours) ---")
print(df_q4.to_string(index=False))

In [ ]:
# Question 5: Calculate hourly utilization for each building
q5 = """
SELECT 
    building_id,
    strftime('%Y-%m-%d %H:00:00', timestamp) as hour_timestamp,
    ROUND(COUNT(CASE WHEN operating_mode = 'NORMAL' THEN 1 END) * 100.0 / COUNT(*), 2) as utilization_percentage
FROM fact_telemetry
WHERE building_id IS NOT NULL AND building_id != ''
GROUP BY building_id, hour_timestamp
LIMIT 5;
"""
df_q5 = pd.read_sql_query(q5, conn)
print("\n--- 5. Hourly Utilization per Building (Sample) ---")
print(df_q5.to_string(index=False))

In [ ]:
# Question 6: Identify sites with abnormal increases in power consumption
q6 = """
WITH daily_site_energy AS (
    SELECT 
        site_id,
        DATE(timestamp) as date_val,
        SUM(power_consumption) * 0.25 as total_energy
    FROM fact_energy
    WHERE site_id IS NOT NULL AND site_id != ''
    GROUP BY site_id, DATE(timestamp)
),
energy_growth AS (
    SELECT 
        site_id,
        date_val,
        total_energy,
        LAG(total_energy) OVER (PARTITION BY site_id ORDER BY date_val) as prev_day_energy
    FROM daily_site_energy
)
SELECT 
    site_id,
    date_val as detection_date,
    ROUND(prev_day_energy, 2) as previous_day_energy_kwh,
    ROUND(total_energy, 2) as current_day_energy_kwh,
    ROUND(((total_energy - prev_day_energy) / prev_day_energy) * 100.0, 2) as percentage_increase
FROM energy_growth
WHERE prev_day_energy IS NOT NULL AND prev_day_energy > 0
  AND total_energy > prev_day_energy * 1.5;
"""
df_q6 = pd.read_sql_query(q6, conn)
print("\n--- 6. Sites with Abnormal Power Spikes (>50% DoD) ---")
print(df_q6.to_string(index=False))

conn.close()

### 4. Multi-Asset Hierarchy & NetworkX Analytics

Using NetworkX, we model the parent-child dependencies of HVAC/chilled-water assets to isolate orphan/disconnected units and evaluate downstream impacts of system failures.

In [ ]:
from src.asset_graph import AssetHierarchyGraph

hierarchy = AssetHierarchyGraph("nectar_iot.db")

print("Orphan Assets (No site/building reference):")
print(hierarchy.get_orphan_assets())

print("\nDisconnected Assets (Site registered, but no graph relationships):")
print(hierarchy.get_disconnected_assets())

print("\nDownstream impact analysis if Chiller-01 (ASSET_001) fails:")
print(hierarchy.get_downstream_impacted("ASSET_001"))